In [9]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Plotting_IQR import plot_distribution, get_training_data, plot_pfn_variance_surface, plot_GP_variance_surface
from pfn_evaluate import eval_pfn
import pfns4bo
from pfns4bo.scripts.acquisition_functions import TransformerBOMethod

In [ ]:
# Load data
file_path = "K_1D_varied_results/run_20260803_172708/metrics.pt"
metrics = torch.load(file_path)

file_path = "K_1D_varied_results/run_20260803_172708/experimental_results.pt"
data = torch.load(file_path)

In [ ]:
#metrics = {
#        "pred_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, n_fns)), # GP & PFN only
#        "total_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, 1)),
#        "EI": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, 1))
#    }
pred_error = metrics["pred_error"]
total_error = metrics["total_error"]
ei = metrics["EI"]
gll = metrics["GLL"]
m_gll = metrics["MGLL"]

y_true_store = data["y_true"][0]
mu_store = data["mu"][0]
var_store = data["var"][0]
x_queried = data["x_query"][0]

In [ ]:
mu_data_GP = torch.sum(mu_store[:, 0, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN = torch.sum(mu_store[:, 1, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN_W = torch.sum(mu_store[:, 2, 0, :, :], dim=-1, keepdim=True)
var_data_GP = torch.sum(var_store[:, 0, 0, :, :], dim=-1, keepdim=True)
var_data_PFN = torch.sum(var_store[:, 1, 0, :, :], dim=-1, keepdim=True)
var_data_PFN_W = torch.sum(var_store[:, 2, 0, :, :], dim=-1, keepdim=True)
x_query_GP = x_queried[:, 0, 0, :, :]
x_query_PFN = x_queried[:, 1, 0, :, :]
x_query_PFN_W = x_queried[:, 2, 0, :, :]
y_true_arr = y_true_store[:, 0, 0, :, :]

kernels = ["Matern12", "Matern32", "Matern52", "RBF"]
roughness = [2, 2/3, 2/5, 0] # inverse of smoothness (nu)

In [20]:
# metrics["pred_error"][test, k, m_idx, rep, :, :]
# 9 tests, 1 dim, 2 methods, 21 reps, 1000 samples, dim = 1 -> 9 tests, 2 methods, 1000 samples, 1 
pred_error_med = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)
total_error_med = torch.quantile(total_error[:, 0, :, :, :], 0.5, dim=-3)
ei_med = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)

pred_error_lq = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)
total_error_lq = torch.quantile(total_error[:, 0, :, :, :], 0.25, dim=-3)
ei_lq = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)

pred_error_uq = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)
total_error_uq = torch.quantile(total_error[:, 0, :, :, :], 0.75, dim=-3)
ei_uq = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)

In [ ]:
gll_med = torch.quantile(torch.sum(gll[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)
mgll_med = torch.quantile(m_gll[:, 0, :, :, :], 0.5, dim=-3)

gll_lq = torch.quantile(torch.sum(gll[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)
mgll_lq = torch.quantile(m_gll[:, 0, :, :, :], 0.25, dim=-3)

gll_uq = torch.quantile(torch.sum(gll[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)
mgll_uq = torch.quantile(m_gll[:, 0, :, :, :], 0.75, dim=-3)

In [ ]:
import numpy as np
import torch

total_error_med = torch.quantile(total_error[:, 0], 0.5, dim=-2).squeeze(-1).cpu().numpy()
total_error_lq  = torch.quantile(total_error[:, 0], 0.25, dim=-2).squeeze(-1).cpu().numpy()
total_error_uq  = torch.quantile(total_error[:, 0], 0.75, dim=-2).squeeze(-1).cpu().numpy()

mgll_med = torch.quantile(m_gll[:, 0], 0.5, dim=-2).squeeze(-1).cpu().numpy()
mgll_lq  = torch.quantile(m_gll[:, 0], 0.25, dim=-2).squeeze(-1).cpu().numpy()
mgll_uq  = torch.quantile(m_gll[:, 0], 0.75, dim=-2).squeeze(-1).cpu().numpy()


kernels = ["Matern12", "Matern32", "Matern52", "RBF"]
roughness = np.array([2.0, 2/3, 2/5, 0.0])

# Get indices that sort roughness from 0.0 (RBF) to 2.0 (Matern12)
sort_idx = np.argsort(roughness)  # Results in array([3, 2, 1, 0])
x_sorted = roughness[sort_idx]
kernels_sorted = [kernels[i] for i in sort_idx]

# Create descriptive tick labels showing both value and kernel name
tick_labels = [f"{val:.2f}\n({name})" for val, name in zip(x_sorted, kernels_sorted)]


methods = ["GP", "PFN", "PFN-W"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
markers = ["o", "s", "^"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

for m_idx, (label, color, marker) in enumerate(zip(methods, colors, markers)):
    # Slice method column and reorder rows by sort_idx
    med = total_error_med[sort_idx, m_idx]
    lq  = total_error_lq[sort_idx, m_idx]
    uq  = total_error_uq[sort_idx, m_idx]
    
    ax1.plot(x_sorted, med, label=label, color=color, marker=marker, 
             linewidth=2, markersize=6)
    ax1.fill_between(x_sorted, lq, uq, color=color, alpha=0.2, 
                     label=f"{label} IQR" if m_idx == 0 else "_nolegend_")

ax1.set_title("Total Absolute Prediction Error vs. Roughness", pad=10)
ax1.set_xlabel("Roughness", fontsize=11)
ax1.set_ylabel("Total Error", fontsize=11)
ax1.set_xticks(x_sorted)
ax1.set_xticklabels(tick_labels)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(frameon=True, facecolor='white', edgecolor='none')

for m_idx, (label, color, marker) in enumerate(zip(methods, colors, markers)):
    med = mgll_med[sort_idx, m_idx]
    lq  = mgll_lq[sort_idx, m_idx]
    uq  = mgll_uq[sort_idx, m_idx]
    
    ax2.plot(x_sorted, med, label=label, color=color, marker=marker, 
             linewidth=2, markersize=6)
    ax2.fill_between(x_sorted, lq, uq, color=color, alpha=0.2)

ax2.set_title("Mean Gaussian Log-Likelihood vs. Roughness", pad=10)
ax2.set_xlabel("Roughness", fontsize=11)
ax2.set_ylabel("MGLL", fontsize=11)
ax2.set_xticks(x_sorted)
ax2.set_xticklabels(tick_labels)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(frameon=True, facecolor='white', edgecolor='none')

plt.tight_layout()
plt.show()